# Collate raw BMS exports

Loads every `cellvoltages_*.csv` under `../data/raw/`, cleans them via
`bms_utils.load_session`, concatenates into a single tidy DataFrame, and
writes `../data/collated.parquet`. Also prints a sampling-rate sanity
check so dropped frames or clock glitches surface here, not three notebooks downstream.

In [1]:
import glob
import pandas as pd
import bms_utils as B

paths = sorted(glob.glob('../data/raw/cellvoltages_*.csv'))
paths

['../data/raw/cellvoltages_2026-03-03-21-21-48.csv',
 '../data/raw/cellvoltages_2026-03-11-15-48-37.csv',
 '../data/raw/cellvoltages_2026-03-27-18-51-29.csv',
 '../data/raw/cellvoltages_2026-03-27-19-01-44.csv',
 '../data/raw/cellvoltages_2026-03-27-19-03-55.csv',
 '../data/raw/cellvoltages_2026-03-27-19-14-37.csv']

**Load and concatenate**

`bms_utils.load_session` strips whitespace, parses the TZ-suffixed timestamps, flips
Pack Current so positive = charging, and tags each row with a `session_id` derived
from the filename. Resistance and Open Cell Voltage columns are kept.

In [2]:
sessions = [B.load_session(p) for p in paths]
if not sessions:
    raise ValueError('No CSV files loaded. Check file paths.')

collated = pd.concat(sessions, ignore_index=True)
collated.to_parquet('../data/collated.parquet')

print(f'rows: {len(collated):,}   cols: {collated.shape[1]}')
print(f'sessions: {collated["session_id"].nunique()}')

rows: 4,814   cols: 269
sessions: 6


**Sampling sanity check**

Each row should arrive ≈1 s after the previous one. A `max_gap_s` much
larger than `p99_dt_s` would indicate dropped frames worth investigating
before drawing conclusions from a derivative metric.

In [3]:
B.sampling_summary(collated)

,session_id,rows,duration_s,median_dt_s,p99_dt_s,max_gap_s
0,2026-03-03-21-21-48,1807,842.0,0.0,2.0,3.0
1,2026-03-11-15-48-37,191,124.0,1.0,2.0,2.0
2,2026-03-27-18-51-29,522,584.0,1.0,2.0,3.0
3,2026-03-27-19-01-44,53,35.0,1.0,2.0,2.0
4,2026-03-27-19-03-55,1342,631.0,0.0,1.0,2.0
5,2026-03-27-19-14-37,899,366.0,0.0,2.0,2.0
